In [3]:
!pip install multiprocess


[notice] A new release of pip is available: 23.2 -> 23.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from instruments.auto_callable_note import AutoCallableNote, ReferenceIndexReturnSimulator

import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm
import numpy as np
import os
import pickle
import multiprocess

In [2]:
def generate_price_grids(start_date: str, end_date: str, model: str = 'SABR'):
    """
    :param start_date:
    :param end_date:
    :param model:
    """
    from instruments.auto_callable_note import AutoCallableNote, ReferenceIndexReturnSimulator

    import pandas as pd
    from tqdm import tqdm
    import numpy as np
    import os
    import pickle

    issue_price = 10
    index_mean = 0
    index_vols = [0.1, 0.3, 0.5, 0.9]
    free_rate = 0.05
    n_paths = 50_000
    delta_s = 0.02
    current_price = [0.1, 1, 3, 4] + np.arange(4, 15, 0.2).tolist() + [15, 17, 19, 30] 
    dates = pd.bdate_range(start_date, end_date)
    
    data = dict()
    for date in tqdm(dates):
        data[f'{date}'] = {}
        for vol in index_vols:
            data[f'{date}'][f'{vol:.2f}'] = {}
            value = []
            delta = []
            gamma = []
            for price in current_price:
                simulator = ReferenceIndexReturnSimulator(issue_price,
                                                          price,
                                                          index_mean, 
                                                          vol,
                                                          sabr_alpha=0.3,
                                                          sabr_beta=1,
                                                          sabr_rho=-0.7,
                                                          delta_s=delta_s,
                                                          model=model)              
                notes = AutoCallableNote(date, simulator, free_rate=free_rate, n_paths=n_paths)  
                value.append(notes.value)
                delta.append(notes.delta)
                gamma.append(notes.gamma)
            data[f'{date}'][f'{vol:.2f}']['stock price'] = current_price
            data[f'{date}'][f'{vol:.2f}']['value'] = value
            data[f'{date}'][f'{vol:.2f}']['delta'] = delta
            data[f'{date}'][f'{vol:.2f}']['gamma'] = gamma
            
    notes_db = {
        'data': data,
        'risk free rate': free_rate
    }
    
    if not os.path.exists('./notes_database/'):
        os.makedirs('./notes_database')
        
    file_name = f'./notes_database/notes_{start_date}_{end_date}.pkl'
    with open(file_name, 'wb') as f:
        pickle.dump(notes_db, f, pickle.HIGHEST_PROTOCOL)
        
    return notes_db


In [3]:
# db = generate_price_gridserate_price_gridserate_price_gridserate_price_grids('2023-09-18', '2024-01-18')

## Create 3 Processes

In [6]:
## Anil

# Please run three times, and remove the comments in different intervals each time
# --------------------------------------------------
# time_period_1 = ('2023-09-18', '2024-01-18', 'SABR')
# time_period_2 = ('2024-01-19', '2024-05-18', 'SABR')
# time_period_3 = ('2024-05-19', '2024-09-18', 'SABR')

# time_period_1 = ('2024-09-19', '2025-01-18', 'SABR')
# time_period_2 = ('2025-01-19', '2025-05-18', 'SABR')
# time_period_3 = ('2025-05-19', '2025-09-18', 'SABR')

# time_period_1 = ('2025-09-19', '2026-01-18', 'SABR')
# time_period_2 = ('2026-01-19', '2026-05-18', 'SABR')
# time_period_3 = ('2026-05-19', '2026-09-18', 'SABR')
# --------------------------------------------------

## Freeman
# --------------------------------------------------
# time_period_1 = ('2026-09-19', '2027-01-18', 'SABR')
# time_period_2 = ('2027-01-29', '2027-05-18', 'SABR')
# time_period_3 = ('2027-05-19', '2027-09-18', 'SABR')

# time_period_1 = ('2027-09-19', '2028-01-18', 'SABR')
# time_period_2 = ('2028-01-29', '2028-05-18', 'SABR')
# time_period_3 = ('2028-05-19', '2028-09-18', 'SABR')

# time_period_1 = ('2028-09-19', '2029-01-18', 'SABR')
# time_period_2 = ('2029-01-19', '2029-05-18', 'SABR')
# time_period_3 = ('2029-05-19', '2029-09-18', 'SABR')

time_period_1 = ('2029-09-19', '2030-01-18', 'SABR')
time_period_2 = ('2030-01-19', '2030-05-18', 'SABR')
time_period_3 = ('2030-05-19', '2030-09-11', 'SABR')
# --------------------------------------------------

p1 = multiprocess.Process(target=generate_price_grids, args=time_period_1)
p2 = multiprocess.Process(target=generate_price_grids, args=time_period_2)
p3 = multiprocess.Process(target=generate_price_grids, args=time_period_3)

# starting process 1
p1.start()
# starting process 2
p2.start()
# starting process 3
p3.start()

# wait until process 1 is finished
p1.join()
# wait until process 2 is finished
p2.join()
p3.join()

# both processes finished
print("Done!")

Done!
